# Module 2: Policy Competition on OpenSpiel

Build 4 policies, compete them on Catch, then switch to another game with the same code.

**Time:** ~20 min · **Difficulty:** Beginner · **GPU:** Not required

In [16]:
# !pip install -q -e ./OpenEnv

import os
import sys

repo = os.path.abspath('OpenEnv')
paths = [repo, os.path.join(repo, 'src'), os.path.join(repo, 'envs')]
for p in paths:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Setup complete!")

Setup complete!


## 1. Connect to Catch

Catch: a ball falls from the top of a 10×5 grid. Move your paddle to catch it.

- Actions: `0` = LEFT, `1` = STAY, `2` = RIGHT
- Reward: `+1` if caught, `0` if missed

In [17]:
from openspiel_env import OpenSpielEnv
from openspiel_env.models import OpenSpielAction, OpenSpielObservation
import random
import requests

CATCH_CANDIDATES = [
    'http://localhost:8000',
    'https://openenv-openspiel-env.hf.space',
]

def pick_openspiel_url(candidates):
    for base_url in candidates:
        try:
            health = requests.get(f"{base_url}/health", timeout=8)
            if health.status_code != 200:
                continue
            metadata = requests.get(f"{base_url}/metadata", timeout=8).json()
            name = str(metadata.get('name', '')).lower()
            if 'openspiel' in name:
                return base_url
        except Exception:
            pass
    return None

CATCH_URL = pick_openspiel_url(CATCH_CANDIDATES)
print(f'Catch URL selected: {CATCH_URL}')

if CATCH_URL is None:
    print('OpenSpiel server not found.')
    print('Start local Catch server on port 8000 and rerun this cell:')
    print('  cd D:/test')
    print('  $env:PYTHONPATH="D:/test/OpenEnv/src;D:/test/OpenEnv/envs"')
    print('  py -m openspiel_env.server.app')
else:
    # Quick sanity check
    with OpenSpielEnv(base_url=CATCH_URL).sync() as env:
        result = env.reset()
        print(f'Legal actions: {result.observation.legal_actions}')
        print(f'Info state shape: {len(result.observation.info_state)} values')
        print(f'Game phase: {result.observation.game_phase}')

Catch URL selected: http://localhost:8000
Legal actions: [0, 1, 2]
Info state shape: 50 values
Game phase: initial


## 2. Define Four Policies

Each policy takes an `OpenSpielObservation` and returns an action ID.

In [18]:
class RandomPolicy:
    """Pure random — baseline."""
    name = "Random"

    def select_action(self, obs: OpenSpielObservation) -> int:
        return random.choice(obs.legal_actions)


class AlwaysStayPolicy:
    """Never moves — hopes ball lands on paddle."""
    name = "Always Stay"

    def select_action(self, obs: OpenSpielObservation) -> int:
        return 1  # STAY


class SmartPolicy:
    """Moves paddle toward ball — optimal for Catch."""
    name = "Smart Heuristic"

    def select_action(self, obs: OpenSpielObservation) -> int:
        info_state = obs.info_state
        grid_width = 5

        # Find ball column (first 1.0 in the flattened grid)
        ball_col = None
        for idx, val in enumerate(info_state):
            if abs(val - 1.0) < 0.01:
                ball_col = idx % grid_width
                break

        # Paddle is in the last row
        last_row = info_state[-grid_width:]
        paddle_col = last_row.index(1.0)

        if ball_col is not None:
            if paddle_col < ball_col:
                return 2  # RIGHT
            elif paddle_col > ball_col:
                return 0  # LEFT
        return 1  # STAY


class LearningPolicy:
    """Epsilon-greedy — starts random, learns to be smart."""
    name = "Epsilon-Greedy"

    def __init__(self):
        self.steps = 0
        self._smart = SmartPolicy()

    def select_action(self, obs: OpenSpielObservation) -> int:
        self.steps += 1
        epsilon = max(0.1, 1.0 - self.steps / 100)
        if random.random() < epsilon:
            return random.choice(obs.legal_actions)
        return self._smart.select_action(obs)


print("Policies defined: Random, Always Stay, Smart Heuristic, Epsilon-Greedy")

Policies defined: Random, Always Stay, Smart Heuristic, Epsilon-Greedy


## 3. Run a Single Episode

Helper to play one full game and return whether the ball was caught.

In [19]:
# Import here so run_episode is self-contained even if cell[3] is skipped
from openspiel_env.models import OpenSpielAction

def run_episode(env, policy, verbose=False):
    """Play one episode. Returns 1 if caught, 0 if missed."""
    result = env.reset()
    step = 0

    while not result.done:
        action_id = policy.select_action(result.observation)
        if verbose:
            name = {0: 'LEFT', 1: 'STAY', 2: 'RIGHT'}.get(action_id, str(action_id))
            print(f'  Step {step}: {name}')
        result = env.step(OpenSpielAction(action_id=action_id, game_name='catch'))
        step += 1

    caught = 1 if result.reward and result.reward > 0 else 0
    if verbose:
        status = 'Caught!' if caught else 'Missed'
        print(f'  Result: {status} (reward={result.reward})')
    return caught


if CATCH_URL is None:
    print('Skipping demo: OpenSpiel server not available.')
else:
    # Demo: one verbose episode with SmartPolicy
    with OpenSpielEnv(base_url=CATCH_URL).sync() as env:
        print('Smart policy — single episode:')
        run_episode(env, SmartPolicy(), verbose=True)

Smart policy — single episode:
  Step 0: STAY
  Step 1: STAY
  Step 2: STAY
  Step 3: STAY
  Step 4: STAY
  Step 5: STAY
  Step 6: STAY
  Step 7: STAY
  Step 8: STAY
  Result: Caught! (reward=1.0)


## 4. Policy Competition

Run 50 episodes per policy and compare success rates.

In [20]:
NUM_EPISODES = 50

policies = [
    RandomPolicy(),
    AlwaysStayPolicy(),
    SmartPolicy(),
    LearningPolicy(),
]

results = {}

if CATCH_URL is None:
    print('Skipping competition: OpenSpiel server not available.')
else:
    with OpenSpielEnv(base_url=CATCH_URL).sync() as env:
        for policy in policies:
            wins = sum(run_episode(env, policy) for _ in range(NUM_EPISODES))
            rate = wins / NUM_EPISODES * 100
            results[policy.name] = rate
            print(f"{policy.name:20s} — {rate:5.1f}% ({wins}/{NUM_EPISODES})")

    print("\n--- Results ---")
    for name, rate in sorted(results.items(), key=lambda x: -x[1]):
        bar = "█" * int(rate / 2)
        print(f"{name:20s} [{bar:<50}] {rate:.1f}%")

Random               —  10.0% (5/50)
Always Stay          —  18.0% (9/50)
Smart Heuristic      — 100.0% (50/50)
Epsilon-Greedy       —  92.0% (46/50)

--- Results ---
Smart Heuristic      [██████████████████████████████████████████████████] 100.0%
Epsilon-Greedy       [██████████████████████████████████████████████    ] 92.0%
Always Stay          [█████████                                         ] 18.0%
Random               [█████                                             ] 10.0%


Expected results:
- **Random**: ~20% (pure luck)
- **Always Stay**: ~20% (terrible strategy)
- **Smart Heuristic**: ~100% (optimal)
- **Epsilon-Greedy**: ~80-90% (improves over episodes)

## 5. Switch Games

The same `OpenSpielEnv` client works for all 6 OpenSpiel games. Let's try Tic-Tac-Toe — the observation format is identical, only the game logic changes.

In [22]:
if CATCH_URL is None:
    print('Skipping Tic-Tac-Toe: OpenSpiel server not available.')
else:
    with OpenSpielEnv(base_url=CATCH_URL).sync() as env:
        result = env.reset()

        # If legal action count is small, server is likely running Catch, not Tic-Tac-Toe.
        if len(result.observation.legal_actions) <= 3:
            print('Current server appears to be Catch (not Tic-Tac-Toe).')
            print('Restart server for Tic-Tac-Toe, then rerun this cell:')
            print('  $env:OPENSPIEL_GAME="tic_tac_toe"')
            print('  py -m openspiel_env.server.app')
        else:
            print(f"Game: Tic-Tac-Toe")
            print(f"Legal actions: {result.observation.legal_actions}")
            print(f"Info state: {result.observation.info_state}")
            print(f"Current player: {result.observation.current_player_id}")
            print()

            # Play randomly until game ends
            step = 0
            while not result.done:
                action_id = random.choice(result.observation.legal_actions)
                result = env.step(OpenSpielAction(action_id=action_id, game_name="tic_tac_toe"))
                step += 1
                print(f"Step {step}: action={action_id}, reward={result.reward}, done={result.done}")

            print(f"\nGame over! Final reward: {result.reward}")

Game: Tic-Tac-Toe
Legal actions: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Info state: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Current player: 0

Step 1: action=6, reward=0.0, done=False
Step 2: action=3, reward=0.0, done=False
Step 3: action=2, reward=0.0, done=False
Step 4: action=8, reward=-1.0, done=True

Game over! Final reward: -1.0


Same client class. Same observation type. Different game. That's the OpenEnv promise.

## Summary

- Built 4 policies with increasing sophistication
- Ran a 50-episode competition on Catch
- Switched to Tic-Tac-Toe with zero code changes to the client

All policies work with `OpenSpielObservation` — you read `info_state`, `legal_actions`, and `done`. The game logic is on the server. Your code is on the client.

**Next:** [Module 3](../module-3/README.md) — Deploying environments to HF Spaces.